# EDA — Dataset muhammadshahidazeem

Dataset de remplacement de rivalytics pour atteindre F1 ≥ 0.70.

**Sources** :
- `data/customer_churn_dataset-training-master.csv` — 440 833 lignes
- `data/customer_churn_dataset-testing-master.csv` — 64 374 lignes

**Objectifs** :
1. Comprendre la structure et la qualité des données
2. Identifier les features discriminantes
3. Définir la stratégie de preprocessing pour la Phase 2

In [ ]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

warnings.filterwarnings('ignore')
sys.path.insert(0, str(Path('..').resolve()))

DATA_DIR = Path('../data')
TARGET = 'Churn'

train = pd.read_csv(DATA_DIR / 'customer_churn_dataset-training-master.csv')
test  = pd.read_csv(DATA_DIR / 'customer_churn_dataset-testing-master.csv')

# Suppression de la ligne nulle (1 null par colonne = ligne header malformée)
train = train.dropna(how='all').dropna(subset=[TARGET])
test  = test.dropna(how='all').dropna(subset=[TARGET])

print(f'Train : {train.shape} | Test : {test.shape}')
train.head(3)

## 1. Structure & qualité des données

In [ ]:
print('=== TYPES ===')
print(train.dtypes)
print('\n=== VALEURS MANQUANTES ===')
print(train.isnull().sum())
print('\n=== DOUBLONS ===')
print(f'CustomerID uniques : {train["CustomerID"].nunique()} / {len(train)}')
print(f'Doublons complets  : {train.duplicated().sum()}')

In [ ]:
print('=== VARIABLES CATÉGORIELLES ===')
for col in ['Gender', 'Subscription Type', 'Contract Length']:
    print(f'\n{col} :')
    print(train[col].value_counts())

In [ ]:
train.describe()

## 2. Distribution de la cible

In [ ]:
churn_counts = train[TARGET].value_counts()
churn_rate = train[TARGET].mean()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].bar(['Non-churn (0)', 'Churn (1)'], churn_counts.values, color=['#2ecc71', '#e74c3c'])
axes[0].set_title('Distribution de la cible (valeurs absolues)')
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 2000, f'{v:,}', ha='center', fontweight='bold')

axes[1].pie(churn_counts.values, labels=['Non-churn', 'Churn'],
            autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'], startangle=90)
axes[1].set_title(f'Taux de churn : {churn_rate:.1%}')

plt.suptitle('Distribution du churn — Dataset muhammadshahidazeem', fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Taux de churn train : {churn_rate:.2%}')
print(f'Taux de churn test  : {test[TARGET].mean():.2%}')

## 3. Features numériques vs churn

In [ ]:
num_features = ['Age', 'Tenure', 'Usage Frequency', 'Support Calls',
                'Payment Delay', 'Total Spend', 'Last Interaction']

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, feat in enumerate(num_features):
    for churn_val, color, label in [(0, '#2ecc71', 'Non-churn'), (1, '#e74c3c', 'Churn')]:
        data = train[train[TARGET] == churn_val][feat].dropna()
        axes[i].hist(data, bins=40, alpha=0.6, color=color, label=label, density=True)
    axes[i].set_title(feat)
    axes[i].legend(fontsize=8)

axes[-1].axis('off')
plt.suptitle('Distributions numériques par classe de churn', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Médianes et test de Mann-Whitney
print(f'{'Feature':<22} {'Médiane Non-churn':>20} {'Médiane Churn':>15} {'p-value':>12}')
print('-' * 72)
for feat in num_features:
    g0 = train[train[TARGET] == 0][feat].dropna()
    g1 = train[train[TARGET] == 1][feat].dropna()
    _, p = stats.mannwhitneyu(g0, g1, alternative='two-sided')
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
    print(f'{feat:<22} {g0.median():>20.2f} {g1.median():>15.2f} {p:>12.2e} {sig}')

## 4. Features catégorielles vs churn

In [ ]:
cat_features = ['Gender', 'Subscription Type', 'Contract Length']

fig, axes = plt.subplots(1, 3, figsize=(14, 5))

for i, feat in enumerate(cat_features):
    churn_by_cat = train.groupby(feat)[TARGET].mean().sort_values(ascending=False)
    bars = axes[i].bar(churn_by_cat.index, churn_by_cat.values,
                        color=['#e74c3c' if v > churn_rate else '#3498db' for v in churn_by_cat.values])
    axes[i].axhline(churn_rate, color='gray', linestyle='--', label=f'Moyenne ({churn_rate:.1%})')
    axes[i].set_title(f'Taux de churn par {feat}')
    axes[i].set_ylabel('Taux de churn')
    axes[i].legend()
    for bar, val in zip(bars, churn_by_cat.values):
        axes[i].text(bar.get_x() + bar.get_width()/2, val + 0.005,
                     f'{val:.1%}', ha='center', fontsize=9)

plt.suptitle('Taux de churn par feature catégorielle', fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Matrice de corrélation

In [ ]:
corr_df = train[num_features + [TARGET]].copy()
corr_matrix = corr_df.corr()

plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.5)
plt.title('Matrice de corrélation — features numériques + Churn', fontweight='bold')
plt.tight_layout()
plt.show()

# Corrélations avec la cible
print('\nCorrélations avec Churn (Pearson) :')
churn_corr = corr_matrix[TARGET].drop(TARGET).abs().sort_values(ascending=False)
for feat, corr in churn_corr.items():
    print(f'  {feat:<22} {corr:.4f}')

## 6. Outliers

In [ ]:
fig, axes = plt.subplots(1, len(num_features), figsize=(18, 5))
for ax, feat in zip(axes, num_features):
    ax.boxplot([train[train[TARGET] == 0][feat].dropna(),
                train[train[TARGET] == 1][feat].dropna()],
               labels=['Non-churn', 'Churn'], patch_artist=True,
               boxprops=dict(facecolor='lightblue'))
    ax.set_title(feat, fontsize=9)
    ax.tick_params(axis='x', labelsize=8)

plt.suptitle('Boxplots — détection des outliers par feature', fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Synthèse & recommandations preprocessing

In [ ]:
print('=' * 60)
print('SYNTHÈSE EDA — Dataset muhammadshahidazeem')
print('=' * 60)

print(f"""
STRUCTURE
  - Train : {len(train):,} lignes, {train.shape[1]} colonnes
  - Test  : {len(test):,}  lignes, {test.shape[1]} colonnes
  - 1 null par colonne (ligne malformée) → dropped

CIBLE
  - Taux de churn : {train[TARGET].mean():.1%} (vs 22% rivalytics)
  - Quasi-équilibré → scale_pos_weight XGBoost moins critique

FEATURES CATÉGORIELLES
  - Gender           : {train['Gender'].nunique()} modalités
  - Subscription Type: {train['Subscription Type'].nunique()} modalités
  - Contract Length  : {train['Contract Length'].nunique()} modalités

PREPROCESSING RECOMMANDÉ
  1. Drop : CustomerID (identifiant, non prédictif)
  2. Encoder Gender → 0/1 (LabelEncoder)
  3. Encoder Subscription Type → ordinal (Basic < Standard < Premium)
     ou one-hot si non-ordinal confirmé par les corrélations
  4. Encoder Contract Length → ordinal (Monthly < Quarterly < Annual)
  5. StandardScaler sur numériques pour LogReg
  6. Pas de jointure multi-tables (dataset plat)
  7. Train/test déjà splitté → utiliser les fichiers fournis
""")